# SloughGPT — SloNet-Native Training

Train the SloughGPT research model with **SloNet** — the pure-NumPy neural network library in `domains.training.slonet`. The entire training, optimization, and generation path runs on NumPy: **no PyTorch required** for the core runtime.

**Pipeline:** `GENERATE_DATA → DISTILL → TRAIN → EVALUATE → DEPLOY → COMPLETE`

**Requirements:**
- Google Colab (free CPU tier works — SloNet is CPU-native), Kaggle, Gradient, or any Jupyter host
- Python 3.9+

**Quick Start:**
1. Runtime → Connect
2. Run cells in order
3. Train with `SloughGPTTrainer.train()` (canonical SloNet-native loop)

**Checkpoints:** written as `.soul` files (self-contained weights + vocab + training state) under `models/auto-training/`, with a `.npz` fallback.

**Reference commands:**
- `make colab-smoke` — smoke-test this notebook (`scripts/run_colab_notebook_smoke.sh`)
- `make colab-test` — run regression tests against the notebook
- `make help` — show all Make targets
- `scripts/run_colab_notebook_smoke.sh --help` — smoke-script options

**Smoke-script install path:** `pip install -e ".[notebook]"` (pulls `jupyter`, `nbclient`, `nbformat`).

## Setup

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DataWeX/sloughGPT.git"

def _ensure_sloughgpt_repo():
    """Move to the sloughGPT repo root, cloning it first if needed."""
    cwd = Path.cwd().resolve()
    if (cwd / "pyproject.toml").is_file():
        return cwd
    repo_dir = cwd / "sloughGPT"
    if (repo_dir / "pyproject.toml").is_file():
        os.chdir(repo_dir)
        return repo_dir
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    return repo_dir

root = _ensure_sloughgpt_repo()
print(f"Repo: {root}")

In [ ]:
# Install SloNet runtime deps (pure numpy — no torch needed).
# Optional notebook tooling extra if you want to run this from nbconvert:
#   !python3 -m pip install -e ".[notebook]"  # needs jupyter, nbclient nbformat
!python3 -m pip install numpy scipy safetensors tqdm requests -q
!python3 -m pip install -e packages/core-py -q 2>/dev/null || true

# Add core-py to sys.path for direct imports
import sys
core_py = Path.cwd() / "packages" / "core-py"
if core_py.is_dir() and str(core_py) not in sys.path:
    sys.path.insert(0, str(core_py))

import numpy as np
print(f"numpy {np.__version__} | core-py OK")

## Runtime

SloNet training is **pure NumPy and always runs on the CPU**. There is no GPU/CUDA/MPS selection — that keeps the notebook lightweight and reproducible on free Colab. See `TrainerConfig.device` (`= "cpu"`) and the Metal-accelerator gating notes in `AGENTS.md` (`_ACCELERATOR` is disabled under `embed_dim ≤ 128`).

Env hooks the smoke script relies on:
- `SLOUGH_NOTEBOOK_TRAIN_CAP` — cap training steps (smoke sets 50)
- `SLOUGH_NOTEBOOK_FORCE_CPU` — force CPU (SloNet is CPU-only anyway; kept for script parity)

In [ ]:
SLOUGH_NOTEBOOK_TRAIN_CAP = int(os.environ.get("SLOUGH_NOTEBOOK_TRAIN_CAP", 1000))
SLOUGH_NOTEBOOK_FORCE_CPU = os.environ.get("SLOUGH_NOTEBOOK_FORCE_CPU", "1").lower() in ("1", "true", "yes")
print(f"Train step cap: {SLOUGH_NOTEBOOK_TRAIN_CAP}")
print(f"Force CPU: {SLOUGH_NOTEBOOK_FORCE_CPU}")

## Dataset

Any UTF-8 text file works. We fetch the classic tiny-Shakespeare corpus for a quick demo.

In [ ]:
import urllib.request

datasets_dir = Path("datasets")
datasets_dir.mkdir(exist_ok=True)
data_path = datasets_dir / "shakespeare" / "input.txt"
data_path.parent.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading tiny-Shakespeare dataset...")
    urllib.request.urlretrieve(url, data_path)

text = data_path.read_text(encoding="utf-8")
print(f"Dataset: {len(text):,} characters at {data_path}")

## Model & Training Configuration

Sizes are conservative so the default run finishes quickly on free CPU. Scale up (`n_embed`, `n_layer`) for higher quality once you verify the loop works. `max_steps` honors `SLOUGH_NOTEBOOK_TRAIN_CAP`.

In [ ]:
MODEL_CONFIG = {
    "n_embed": 96,       # embedding dim
    "n_layer": 4,        # transformer blocks
    "n_head": 8,         # must divide n_embed
    "block_size": 96,    # context window
    "dropout": 0.1,
}

TRAIN_CONFIG = {
    "batch_size": 32,
    "epochs": 3,
    "learning_rate": 1e-3,
    "max_steps": min(1500, SLOUGH_NOTEBOOK_TRAIN_CAP),
    "gradient_accumulation_steps": 1,
    "max_grad_norm": 1.0,
    "weight_decay": 0.01,
    "warmup_steps": 50,
    "scheduler_type": "cosine",
    "checkpoint_dir": str(Path("models/auto-training")),
}

for k, v in {**MODEL_CONFIG, **TRAIN_CONFIG}.items():
    print(f"  {k}: {v}")

## Initialize Trainer

`prepare_data` returns token ids as a NumPy int64 array plus the char<->id maps. `SloughGPTTrainer` embeds vocab + personality into the model, so every model is a soul.

In [ ]:
from domains.training.train_pipeline import SloughGPTTrainer, TrainerConfig, prepare_data

print("Preparing data...")
data, vocab_size, stoi, itos = prepare_data(str(data_path), block_size=MODEL_CONFIG["block_size"])
print(f"Vocabulary: {vocab_size} unique characters")

trainer_config = TrainerConfig(
    vocab_size=vocab_size,
    n_embed=MODEL_CONFIG["n_embed"],
    n_layer=MODEL_CONFIG["n_layer"],
    n_head=MODEL_CONFIG["n_head"],
    block_size=MODEL_CONFIG["block_size"],
    dropout=MODEL_CONFIG["dropout"],
    batch_size=TRAIN_CONFIG["batch_size"],
    epochs=TRAIN_CONFIG["epochs"],
    learning_rate=TRAIN_CONFIG["learning_rate"],
    max_steps=TRAIN_CONFIG["max_steps"],
    gradient_accumulation_steps=TRAIN_CONFIG["gradient_accumulation_steps"],
    max_grad_norm=TRAIN_CONFIG["max_grad_norm"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
    warmup_steps=TRAIN_CONFIG["warmup_steps"],
    scheduler_type=TRAIN_CONFIG["scheduler_type"],
    checkpoint_dir=TRAIN_CONFIG["checkpoint_dir"],
    device="cpu",
)

# Override with pre-parsed data so the trainer does not re-read the file.
trainer = SloughGPTTrainer(data_path=str(data_path), config=trainer_config)
trainer.data = data
trainer.vocab_size = vocab_size
trainer.stoi = stoi
trainer.itos = itos

num_params = trainer.model.num_parameters()
print(f"Model parameters: {num_params:,} ({num_params/1e6:.2f}M)")

## Train (SloNet-native)

This calls the canonical `trainer.train()` loop — gradient accumulation, EMA-smoothed loss reporting, LR scheduling, and automatic `.soul` checkpointing with rotation. No manual torch optimizer/scheduler.

In [ ]:
import time
from IPython.display import clear_output

print(f"Training: device=cpu | steps={trainer_config.max_steps} | batch={trainer_config.batch_size}")
start_time = time.time()

def _on_progress(info: dict) -> None:
    # info: global_step, epoch, epochs, steps_per_epoch, progress_percent,
    #       train_loss (may be None on the first step), learning_rate
    if info.get("global_step", 0) % 25 != 0:
        return
    step = info["global_step"]
    loss = info.get("train_loss")
    loss_s = f"{loss:.4f}" if loss is not None else "n/a"
    elapsed = time.time() - start_time
    print(f"step {step:5d} (epoch {info.get('epoch', 0)}/{info.get('epochs', 0)})"
          f" | loss {loss_s} | lr {info.get('learning_rate', float('nan')):.2e} | {elapsed:.1f}s")
    clear_output(wait=True)

result = trainer.train(on_progress=_on_progress)

minutes = (time.time() - start_time) / 60
print(f"\nTraining complete in {minutes:.1f} minutes")
print(f"Final train loss: {result.final_loss}")
print(f"Steps: {result.global_step} | method: {result.method}")

## Generate Text

`trainer.generate()` handles tokenization, KV-cached generation, temperature sampling, and decoding — pure NumPy.

In [ ]:
prompt = "First Citizen:"
print(f"Prompt: {prompt}")
print("Generated:\n")
print(trainer.generate(prompt, max_tokens=200, temperature=0.8))

## Save / Export

The trainer already wrote `.soul` checkpoints under `models/auto-training/`. Here we also write an explicit `.npz` export and (in Colab) download it.

In [ ]:
from domains.training.export import export_to_sou

output_dir = Path("models/colab_export")
output_dir.mkdir(parents=True, exist_ok=True)

sou_path = output_dir / "sloughgpt.soul"
export_to_sou(trainer.model, str(sou_path), soul_profile=None, weights_only=True)
print(f".soul: {sou_path}")

npz_path = output_dir / "sloughgpt.npz"
np.savez(str(npz_path), **{name: np.asarray(param) for name, param in trainer.model.state_dict().items()})
print(f".npz: {npz_path}")

auto_checkpoints = sorted(Path(TRAIN_CONFIG["checkpoint_dir"]).glob("*.soul"))
print("\nAuto checkpoints saved during training:")
for c in auto_checkpoints:
    print(f"  {c.name} ({c.stat().st_size/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(str(npz_path))
except ImportError:
    print("(Run in Google Colab to trigger a download)")

## Quantize for CPU Inference

SloNet ships **AVX-512 VNNI / AVX2-accelerated int8/int4 GEMM kernels** ("llama.cpp-style") in `domains/infrastructure/quant_core`. The int8 kernel uses the fused `_mm512_dpbusd_epi32` instruction (one dot-product per 512-bit ZMM lane) when the CPU supports AVX-512 BW+VNNI, falling back to AVX2, then scalar. This notebook applies **adaptive quantization**: `apply_adaptive_quantization` quantizes a layer with int8 only when that layer is large enough for the int8 kernel to beat numpy fp32. The crossover inner dim is ~512 with AVX-512 VNNI (the faster kernel), ~1024 with AVX2. `should_quantize_row` exposes the per-layer decision, `HAS_AVX2` reports the AVX2 kernel, and `HAS_AVX512` reports whether the AVX-512 VNNI path is active. Small-embed layers below the crossover stay on fp32, so quantization never regresses a small model.

In [ ]:
from domains.infrastructure.quantization import apply_adaptive_quantization, should_quantize_row
from domains.infrastructure.quant_core.wrapper import HAS_AVX2, HAS_AVX512
import time

print(f"AVX2 quantized GEMM available: {HAS_AVX2} | AVX-512 VNNI path: {HAS_AVX512}")

# 1. Quantize adaptively: apply int8 only to layers where it beats fp32.
#    On this host numpy fp32 runs 512-bit AVX512 FMA, so the AVX2 int8 kernel
#    only wins for large weight matrices (inner dim K >= QUANT_CROSSOVER_K); the
#    AVX-512 VNNI kernel lowers that crossover to ~512.
#    Small-embed layers are left on fp32 so quantization never regresses them.
res = apply_adaptive_quantization(trainer.model, bits=8, mode="symmetric")
q = res['quantized']; t = res['total']; lf = res['left_fp32']
print(f"Quantized {q}/{t} linear layers (int8); {lf} left on fp32 (below int8 crossover)")

# 2. Bench final inference once.
trainer.generate("First Citizen:", max_tokens=8, temperature=0.8)  # warmup
n_tokens = 64
start = time.perf_counter()
trainer.generate("First Citizen:", max_tokens=n_tokens, temperature=0.8)
ms = (time.perf_counter() - start) * 1000
path = 'AVX2 int8' if q else 'fp32'
print(f"{path} generation ({n_tokens} tokens): {ms:.1f} ms")
if q == 0:
    print('Small-embed model: int8 would not beat fp32 here, so all layers kept fp32.')
    print('Raise n_embed (config cell above) to >= crossover to see int8 win.')

# 3. Sanity: generation still works through the active path.
print("Sample:", trainer.generate("First Citizen:", max_tokens=80, temperature=0.8))


## Resume Training

Continue from the newest checkpoint (picks the latest `.soul` under the checkpoint dir).

In [ ]:
# Pass resume=True to continue from the latest auto-checkpoint.
resumed = trainer.train(resume=True, on_progress=_on_progress)
print("Resumed training finished.")

## Cognitive Helpers

Small async utilities for driving coroutine-based features (e.g. streaming) from a notebook.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

def _asyncio_run(coro):
    """Run a coroutine from synchronous notebook context."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(max_workers=2) as executor:
        return executor.submit(asyncio.run, coro).result()

print("Cognitive helpers loaded")

## Serve the Trained Model

The SloughGPT stack runs on **FastAPI + uvicorn** (see `apps/api/server/main.py`), not Flask. Start the full API server from the notebook with a background process:

- Then hit `http://localhost:8000` with the bundled SDK or `curl`.
- Example: `!python3 -m uvicorn apps.api.server.main:app --host 0.0.0.0 --port 8000`

In [ ]:
# Start the FastAPI server in the background (uvicorn is the repo's runtime).
# This runs for the lifetime of the kernel; stop it with OS.kill[pid] if needed.
import subprocess, os, signal

proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "apps.api.server.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"FastAPI server PID {proc.pid} on :8000")
print("Stop it later with: os.kill(%d, signal.SIGTERM)" % proc.pid)

# To stop:
# os.kill(proc.pid, signal.SIGTERM)

## Bonus: Flask (optional)

A notebook **can** also run Flask, but Flask is **not** a dependency of this repository — the repo's web stack is FastAPI/uvicorn. If you specifically want a Flask endpoint for the trained model, install it and serve the in-kernel trainer:


In [ ]:
# Optional — Flask is NOT part of the repo. Install it only if you want this.
# !python3 -m pip install flask -q

# from flask import Flask, request, jsonify
# app = Flask(__name__)

# @app.post("/api/generate")
# def generate():
#     body = request.get_json(force=True)
#     text = trainer.generate(body.get("prompt", ""), max_tokens=body.get("max_tokens", 100))
#     return jsonify({"generated": text})

# app.run(host="0.0.0.0", port=5000)
print("Flask snippet is optional and commented out — see the full API server above.")

## Next Steps

- **Scale up:** raise `n_embed`/`n_layer`/`epochs` and lose the `max_steps` cut.
- **Resume:** training already auto-checkpoints `.soul` files; `trainer.train(resume=True)` continues.
- **Distill / RLHF:** the [`TrainingSequence`](packages/core-py/domains/training/sequence.py) protocol drives GENERATE_DATA → DISTILL → TRAIN → EVALUATE → DEPLOY → COMPLETE.
- **CLI:** `sloughgpt train quick`, `sloughgpt train self` etc. (see `apps/cli`).
- **Serve:** FastAPI server on :8000, or export `.soul` and load it via the SloNet runtime.

Verify with `make colab-test`, or run the whole notebook headlessly with `scripts/run_colab_notebook_smoke.sh`.